# SHA-256
“merhaba” örneği ve ilk bloktaki bütün temel adımlar.


In [ ]:
"""SHA-256 eğitim uygulaması: hashlib sonucu ve ilk blok için ara değerler."""
import hashlib

K = [
0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,
0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf174,
0xe49b69c1,0xefbe4786,0x0fc19dc6,0x240ca1cc,0x2de92c6f,0x4a7484aa,0x5cb0a9dc,0x76f988da,
0x983e5152,0xa831c66d,0xb00327c8,0xbf597fc7,0xc6e00bf3,0xd5a79147,0x06ca6351,0x14292967,
0x27b70a85,0x2e1b2138,0x4d2c6dfc,0x53380d13,0x650a7354,0x766a0abb,0x81c2c92e,0x92722c85,
0xa2bfe8a1,0xa81a664b,0xc24b8b70,0xc76c51a3,0xd192e819,0xd6990624,0xf40e3585,0x106aa070,
0x19a4c116,0x1e376c08,0x2748774c,0x34b0bcb5,0x391c0cb3,0x4ed8aa4a,0x5b9cca4f,0x682e6ff3,
0x748f82ee,0x78a5636f,0x84c87814,0x8cc70208,0x90befffa,0xa4506ceb,0xbef9a3f7,0xc67178f2]
H0=[0x6a09e667,0xbb67ae85,0x3c6ef372,0xa54ff53a,0x510e527f,0x9b05688c,0x1f83d9ab,0x5be0cd19]
MASK=0xffffffff

def rotr(x,n): return ((x>>n)|(x<<(32-n)))&MASK
def pad(data):
    bit_length=len(data)*8
    result=data+b'\x80'
    result+=b'\x00'*((56-len(result)%64)%64)
    return result+bit_length.to_bytes(8,'big')

def trace_first_block(text="merhaba"):
    data=text.encode('utf-8'); padded=pad(data); block=padded[:64]
    words=[int.from_bytes(block[i:i+4],'big') for i in range(0,64,4)]
    for t in range(16,64):
        s0=rotr(words[t-15],7)^rotr(words[t-15],18)^(words[t-15]>>3)
        s1=rotr(words[t-2],17)^rotr(words[t-2],19)^(words[t-2]>>10)
        words.append((words[t-16]+s0+words[t-7]+s1)&MASK)
    a,b,c,d,e,f,g,h=H0; rounds=[]
    for t in range(64):
        S1=rotr(e,6)^rotr(e,11)^rotr(e,25); ch=(e&f)^((~e)&g)
        temp1=(h+S1+ch+K[t]+words[t])&MASK
        S0=rotr(a,2)^rotr(a,13)^rotr(a,22); maj=(a&b)^(a&c)^(b&c)
        temp2=(S0+maj)&MASK
        before=[a,b,c,d,e,f,g,h]
        h,g,f,e,d,c,b,a=g,f,e,(d+temp1)&MASK,c,b,a,(temp1+temp2)&MASK
        rounds.append(dict(t=t,before=before,w=words[t],k=K[t],S1=S1,ch=ch,temp1=temp1,S0=S0,maj=maj,temp2=temp2,after=[a,b,c,d,e,f,g,h]))
    state=[(x+y)&MASK for x,y in zip(H0,[a,b,c,d,e,f,g,h])]
    return dict(text=text,data=list(data),padded=list(padded),words=words,rounds=rounds,state=state,digest=''.join(f'{x:08x}' for x in state))

def sha256_text(text): return hashlib.sha256(text.encode('utf-8')).hexdigest()


## 1. Doğrudan SHA-256 özeti
Önce Python standart kütüphanesiyle sonucu görelim.


In [ ]:
text = "merhaba"
digest = sha256_text(text)
print("Metin :", text)
print("SHA-256:", digest)
print("Hex karakter sayısı:", len(digest))

Metin : merhaba
SHA-256: 4c6bcdd55f3153e1939669ab1ec039e4059174dc25abdfcb2f58868849b4d61b
Hex karakter sayısı: 64


## 2. Metni baytlara çevir
SHA-256 metni değil, bayt dizisini işler.


In [ ]:
data = text.encode("utf-8")
print("Baytlar:", list(data))
print("Hex    :", data.hex(" "))
print("Bit uzunluğu:", len(data) * 8)

Baytlar: [109, 101, 114, 104, 97, 98, 97]
Hex    : 6d 65 72 68 61 62 61
Bit uzunluğu: 56


## 3. Dolgu uygula
Bir 1 biti, yeterli sıfır ve en sona 64 bitlik mesaj uzunluğu eklenir.


In [ ]:
padded = pad(data)
print("Dolgulu uzunluk:", len(padded), "bayt")
print("Dolgulu blok:", padded.hex(" "))
print("Son 8 bayt:", padded[-8:].hex(" "))

Dolgulu uzunluk: 64 bayt
Dolgulu blok: 6d 65 72 68 61 62 61 80 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 38
Son 8 bayt: 00 00 00 00 00 00 00 38


## 4. İlk 16 kelime
512 bitlik blok, 16 adet 32 bitlik kelimeye ayrılır.


In [ ]:
trace = trace_first_block(text)
for index, word in enumerate(trace["words"][:16]):
    print(f"W{index:02} = {word:08x}")

W00 = 6d657268
W01 = 61626180
W02 = 00000000
W03 = 00000000
W04 = 00000000
W05 = 00000000
W06 = 00000000
W07 = 00000000
W08 = 00000000
W09 = 00000000
W10 = 00000000
W11 = 00000000
W12 = 00000000
W13 = 00000000
W14 = 00000000
W15 = 00000038


## 5. Mesaj planını 64 kelimeye genişlet
W16–W63 önceki kelimelerden üretilir.


In [ ]:
for index, word in enumerate(trace["words"][16:], 16):
    print(f"W{index:02} = {word:08x}")

W16 = 01f40313
W17 = 617d6180
W18 = 81eb9dc4
W19 = 1ce863c9
W20 = bd7aaa2f
W21 = 3d9a97f1
W22 = 007d77f0
W23 = 1afde2ee
W24 = 768380ec
W25 = cf180f1d
W26 = cd5e793a
W27 = c3d9429b
W28 = 3124008b
W29 = 89abe7b7
W30 = 0b649aab
W31 = 2c8c8856
W32 = 039b9ed4
W33 = 825650a8
W34 = 8bcbe33c
W35 = 12cc7898
W36 = 1535e101
W37 = ba4e3ecb
W38 = 2181a6b1
W39 = 2d32265f
W40 = 021ae868
W41 = a552f291
W42 = 88bee3e6
W43 = 10a6dab4
W44 = 20ebc399
W45 = d306cb12
W46 = 5e2b52db
W47 = 3b48a710
W48 = 4174258a
W49 = e31bc6c5
W50 = 5f3106de
W51 = 480759ff
W52 = 9b6f8989
W53 = 702b5e76
W54 = c5095ec4
W55 = 9dedb10b
W56 = f506fa82
W57 = d806c32a
W58 = 4f8b364c
W59 = 2c9782be
W60 = 1c15cf3d
W61 = 32e40b83
W62 = 6972b85f
W63 = cadbe465


## 6. İlk turu incele
Sekiz durum kelimesi T1 ve T2 ile güncellenir.


In [ ]:
round0 = trace["rounds"][0]
for key in ["w", "k", "S1", "ch", "temp1", "S0", "maj", "temp2"]:
    print(f"{key:5} = {round0[key]:08x}")
print("Tur çıkışı:", " ".join(f"{x:08x}" for x in round0["after"]))

w     = 6d657268
k     = 428a2f98
S1    = 3587272b
ch    = 1f85c98c
temp1 = 60dd5fd0
S0    = ce20b47e
maj   = 3a6fe667
temp2 = 08909ae5
Tur çıkışı: 696dfab5 6a09e667 bb67ae85 3c6ef372 062d550a 510e527f 9b05688c 1f83d9ab


## 7. 64 turun sonucu
Son durum başlangıç değerlerine eklenir.


In [ ]:
print("Son durum:", " ".join(f"{x:08x}" for x in trace["state"]))
print("Özet:", trace["digest"])
assert trace["digest"] == sha256_text(text)

Son durum: 4c6bcdd5 5f3153e1 939669ab 1ec039e4 059174dc 25abdfcb 2f588688 49b4d61b
Özet: 4c6bcdd55f3153e1939669ab1ec039e4059174dc25abdfcb2f58868849b4d61b


## 8. Küçük değişiklik, farklı özet
Tek karakter değiştirildiğinde özet bütünüyle değişir.


In [ ]:
for value in ["merhaba", "Merhaba", "merhaba!"]:
    print(value, "→", sha256_text(value))

merhaba → 4c6bcdd55f3153e1939669ab1ec039e4059174dc25abdfcb2f58868849b4d61b
Merhaba → 7fdc9f4717c5fe66df286c700fab969b4d6209d03aa84624c5f8f58c17c9c058
merhaba! → f1b3d7c4f9bf47d412ee584081ca5eda31b9af1682a23e988d46a3376542180f
